# Factors: association and uncertainty

Run from an environment with the project installed after migrations and real market-data import. Load real provider observations before executing this notebook.


In [ ]:
import os
from pathlib import Path

if Path.cwd().name == 'notebooks':
    os.chdir('..')
import pandas as pd
from sqlalchemy import select

from app.db.database import SessionLocal
from app.db.models import Dataset
from app.research.returns import simple_returns
from app.services.research import price_panel

db=SessionLocal()
dataset=db.scalar(select(Dataset).where(~Dataset.source.ilike("%SYNTHETIC%")).order_by(Dataset.created_at.desc()))
assert dataset is not None, 'Run migrations and ingest_market_data first'
panel=price_panel(db,dataset.id,['AAPL','MSFT','SPY'])
returns=simple_returns(panel)
print('Source:',dataset.source,'Dataset:',dataset.id)

In [ ]:
try:
    from app.research.factors import regress
    from app.services.research import factor_panel
    
    f,rf=factor_panel(db,dataset.id,returns.index,['market','size','value','momentum'])
    result=regress(returns.AAPL,f,rf)
    print(result['r_squared'])
    pd.DataFrame(result['coefficients']).T
except ValueError as exc:
    print(f"Factor analysis unavailable: {exc}. Import genuine factor and risk-free data first.")


## Interpretation

Inspect coefficient intervals. Use only genuine dated factor returns; factor data is not supplied by the price API.


In [ ]:
db.close()